In [171]:
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error,r2_score

In [172]:
df=pd.read_csv('/content/seller_recommender_dataset_joel.csv')
df.head()

,main_category,sub_category,ratings,no_of_ratings,discount_price,actual_price,Price,popularity_score,rating_category,discount_percentage,...,sub_category_Camera Accessories,sub_category_Cameras,sub_category_Home Entertainment Systems,sub_category_Personal Care Appliances,sub_category_Refrigerators,sub_category_Security Cameras,sub_category_Speakers,sub_category_Televisions,ratings_price_interaction,discount_popularity_interaction
0,beauty & health,Personal Care Appliances,-0.506505,-0.491014,598.0,899.0,598.0,-1.119931,Average,-0.536745,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,24.488621,2.345482
1,beauty & health,Personal Care Appliances,0.298246,-0.438876,3499.0,3599.0,3499.0,-0.395118,High,-2.029183,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,33.573625,0.315855
2,beauty & health,Personal Care Appliances,0.942046,-0.409911,809.0,1380.0,809.0,-0.007528,High,-0.152971,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,32.537534,5.668768
3,beauty & health,Personal Care Appliances,0.298246,1.229541,700.0,999.0,700.0,1.610707,High,-0.709389,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,28.321797,7.015520
4,beauty & health,Personal Care Appliances,-1.472206,-0.427290,349.0,1599.0,349.0,-0.843148,Low,1.635686,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,22.133277,6.778545


In [173]:
df.tail()

,main_category,sub_category,ratings,no_of_ratings,discount_price,actual_price,Price,popularity_score,rating_category,discount_percentage,...,sub_category_Camera Accessories,sub_category_Cameras,sub_category_Home Entertainment Systems,sub_category_Personal Care Appliances,sub_category_Refrigerators,sub_category_Security Cameras,sub_category_Speakers,sub_category_Televisions,ratings_price_interaction,discount_popularity_interaction
72331,"tv, audio & cameras",Cameras,-0.023654,1.270093,1414.0,2799.0,1414.0,1.435479,Average,0.241010,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,30.955761,11.076631
72332,"tv, audio & cameras",Cameras,-1.150305,-0.491014,4049.0,7700.0,3914.0,-1.249258,Average,0.140566,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,28.637138,2.952527
72333,"tv, audio & cameras",Cameras,0.459196,-0.351980,11990.0,22990.0,11990.0,0.112500,High,0.161531,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,42.180004,6.900836
72334,"tv, audio & cameras",Cameras,0.137296,-0.351980,5249.0,10497.0,5249.0,-0.001613,Average,0.265960,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,37.035760,6.867320
72335,"tv, audio & cameras",Cameras,1.102997,-0.328807,7967.0,7967.0,7967.0,0.433482,High,-2.164244,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,41.322668,0.000000


In [174]:
df.isna().sum()

,0
main_category,0
sub_category,0
ratings,0
no_of_ratings,0
discount_price,0
...,...
sub_category_Security Cameras,0
sub_category_Speakers,0
sub_category_Televisions,0
ratings_price_interaction,0


In [175]:
target = "Price"
categorical_features = ["main_category", "sub_category", "rating_category", "price_bucket"]

In [176]:
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [177]:
df["log_price"] = np.log1p(df["Price"])
df["log_discount_price"] = np.log1p(df["discount_price"])
df["log_actual_price"] = np.log1p(df["actual_price"])

In [178]:
df_sampled = df.sample(frac=0.3, random_state=42)

In [179]:
X = df_sampled.drop(columns=[target, "Price"])
y = df_sampled["log_price"]

In [180]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [181]:
y_train_cleaned = y_train.replace([np.inf, -np.inf], np.nan).dropna()
X_train_cleaned = X_train.loc[y_train_cleaned.index]

In [182]:
y_train_cleaned = y_train_cleaned[np.abs(y_train_cleaned) < 1e6]  # Remove excessively large values
X_train_cleaned = X_train_cleaned.loc[y_train_cleaned.index]


In [183]:
X_train_cleaned = X_train_cleaned.replace([np.inf, -np.inf], np.nan).dropna()
y_train_cleaned = y_train_cleaned.loc[X_train_cleaned.index]

In [184]:
assert not y_train_cleaned.isnull().any(), "y_train_cleaned still contains NaN"
assert np.isfinite(y_train_cleaned).all(), "y_train_cleaned still contains infinite values"
assert np.isfinite(X_train_cleaned.values).all(), "X_train_cleaned contains invalid values"

In [185]:
dtrain = xgb.DMatrix(X_train_cleaned, label=y_train_cleaned)
dtest = xgb.DMatrix(X_test, label=y_test)

In [186]:
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.1,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}


In [187]:
num_boost_round = 100
model = xgb.train(params, dtrain, num_boost_round)

In [188]:
y_pred = model.predict(dtest)

In [189]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
map = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [190]:
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R2 Score: {r2}")

RMSE: 0.7374944644616317
MAE: 0.08768956552708237
R2 Score: 0.8972952478465801
